# OnionEye · 01 · Get the data
Downloads all 8 public datasets and builds **Dataset 1 (finder: onion + coin)**. No GPU needed, about 20 min.

Before running: Roboflow account → Settings → API Keys → copy the private key → Colab **Secrets** (key icon, left bar) → name `ROBOFLOW_API_KEY`, turn on notebook access. Then **Runtime → Run all**.

Output in `MyDrive/onioneye/`: `finder_dataset.zip`, `raw_defect_sources.zip`, `mendeley_onion.zip`, `reports/finder_samples.png`.

In [ ]:
!pip -q install roboflow pyyaml opencv-python-headless

In [ ]:
# ---- setup: works on Google Colab (default) and Kaggle
import os, sys, glob, shutil, zipfile, random
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive/onioneye', '/content'
elif IN_KAGGLE:
    ROOT, WORK = '/kaggle/working/onioneye', '/kaggle/temp'
else:
    ROOT, WORK = os.path.abspath('onioneye_data'), os.path.abspath('work')
for d in (ROOT, WORK, f'{ROOT}/models', f'{ROOT}/reports'):
    os.makedirs(d, exist_ok=True)
print('saving to', ROOT)

In [ ]:
# the shared data code lives in your Drive: MyDrive/onioneye/onioneye_data.py (copy it from the repo's ml/ folder)
shutil.copy(f'{ROOT}/onioneye_data.py', 'onioneye_data.py')
import importlib, onioneye_data as od
importlib.reload(od)

In [ ]:
# Roboflow key from Secrets (Colab: key icon on the left, name ROBOFLOW_API_KEY, notebook access ON)
def get_roboflow_key():
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get('ROBOFLOW_API_KEY')
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    return os.environ['ROBOFLOW_API_KEY']
ROBOFLOW_API_KEY = get_roboflow_key()
assert ROBOFLOW_API_KEY, 'Add ROBOFLOW_API_KEY in Secrets first'

## 1 · Download from Roboflow (latest version of each)

In [ ]:
import onioneye_data as od
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
RAW = f'{WORK}/raw'
failed = []
for src in od.FINDER_SOURCES + od.FINDER_TEST_SOURCES + od.DEFECT_BOX_SOURCES:
    try: od.rf_download(rf, src, RAW, 'yolov8')
    except Exception as e: failed.append(src['name']); print('  FAILED', src['name'], e)
for src in od.DEFECT_FOLDER_SOURCES:
    try: od.rf_download(rf, src, RAW, 'folder')
    except Exception as e: failed.append(src['name']); print('  FAILED', src['name'], e)
print('failed:', failed or 'none')

## 2 · Build Dataset 1 (finder)

In [ ]:
import pandas as pd
FINDER = f'{WORK}/finder'
stats, unmapped, dropped = od.build_finder_dataset(od.FINDER_SOURCES, od.FINDER_TEST_SOURCES, RAW, FINDER)
df = pd.Series(stats).unstack(0).fillna(0).astype(int)
print('images per split:', {sp: len(os.listdir(f'{FINDER}/images/{sp}')) for sp in ['train', 'val', 'test', 'test_unseen']})
display(df)
print('dropped images with no usable label:', dropped)
if unmapped: print('⚠ source classes not in the map (tell Claude):', dict(unmapped))

## 3 · Look at 24 random labelled images: green = onion, white = coin

In [ ]:
import cv2, numpy as np
def draw(im_path):
    img = cv2.imread(im_path); H, W = img.shape[:2]
    for ln in open(od.label_path(im_path)).read().splitlines():
        c, *v = ln.split(); pts = (np.array(v, float).reshape(-1, 2) * [W, H]).astype(np.int32)
        cv2.polylines(img, [pts], True, (60, 200, 60) if c == '0' else (255, 255, 255), 3)
    return img
picks = random.sample(glob.glob(f'{FINDER}/images/train/*'), 24)
tmp = [f'{WORK}/_s{i}.jpg' for i in range(24)]
for p, t in zip(picks, tmp): cv2.imwrite(t, draw(p))
sheet = od.contact_sheet(tmp, f'{ROOT}/reports/finder_samples.png', cols=6, cell=260, title='finder dataset')
from IPython.display import Image; display(Image(sheet))

## 4 · Save to Drive

In [ ]:
shutil.make_archive(f'{ROOT}/finder_dataset', 'zip', FINDER)
defect_raw = f'{WORK}/raw_defects'
os.makedirs(defect_raw, exist_ok=True)
for s in od.DEFECT_BOX_SOURCES + od.DEFECT_FOLDER_SOURCES:
    if os.path.exists(f'{RAW}/{s["name"]}'): shutil.copytree(f'{RAW}/{s["name"]}', f'{defect_raw}/{s["name"]}', dirs_exist_ok=True)
shutil.make_archive(f'{ROOT}/raw_defect_sources', 'zip', defect_raw)
print(os.listdir(ROOT))

## 5 · Mendeley bulbs (1.6 GB, straight to your Drive, 5–15 min). Only needed for notebook 03.

In [ ]:
try:
    MZIP = od.mendeley_download(f'{ROOT}/mendeley_onion.zip')
    print('OK', MZIP, round(os.path.getsize(MZIP) / 1e9, 2), 'GB')
except Exception as e:
    print('⚠ Mendeley download failed, the rest is fine. Fix before notebook 03:', e)

Done ✅ Next: **02_train_finder** (switch the runtime to a T4 GPU).